# 14.12 - Agent Evaluation

Status: VERIFIED

## What Are We Solving?
Measuring agent performance requires more than checking the final answer — you must evaluate tool selection, reasoning steps, efficiency, and cost.

In [1]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # loads from .env in project root
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "qwen/qwen3.8-27b"

# Quick test
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'groq ok' only"}],
    max_tokens=10
)
print(f"Groq connected: {r.choices[0].message.content.strip()}")
print(f"Model: {MODEL}")

Groq connected: groq ok
Model: qwen/qwen3.8-27b


## Agent Trajectory Evaluation

In [2]:
from dataclasses import dataclass, field

@dataclass
class AgentTrajectory:
    task: str
    steps: list = field(default_factory=list)
    final_result: str = ""
    total_tokens: int = 0
    
    def add_step(self, tool: str, input_text: str, output_text: str):
        self.steps.append({"tool": tool, "input": input_text[:80], "output": output_text[:80]})
    
    def evaluate(self, expected: str) -> dict:
        return {
            "completed": bool(self.final_result),
            "steps_taken": len(self.steps),
            "exact_match": self.final_result.strip().lower() == expected.strip().lower(),
            "total_tokens": self.total_tokens,
        }

# Run agent and capture trajectory
def run_agent_with_trace(task: str) -> AgentTrajectory:
    traj = AgentTrajectory(task=task)
    messages = [
        {"role": "system", "content": "Answer the question. Be concise."},
        {"role": "user", "content": task}
    ]
    
    tools = [{"type": "function", "function": {"name": "search", "description": "Search for info", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}]
    
    for step in range(5):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, tool_choice="auto")
        msg = response.choices[0].message
        traj.total_tokens += response.usage.total_tokens if response.usage else 0
        
        if msg.tool_calls:
            messages.append(msg)
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                result = f"Info about: {args.get('query', '')}"
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
                traj.add_step("search", json.dumps(args), result)
        else:
            traj.final_result = msg.content
            traj.add_step("final_answer", task, msg.content[:80])
            break
    
    return traj

# Evaluate
traj = run_agent_with_trace("What is gradient descent?")
metrics = traj.evaluate("Gradient descent is an optimization algorithm.")
print("Trajectory Evaluation:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

Trajectory Evaluation:
  completed: True
  steps_taken: 2
  exact_match: False
  total_tokens: 851


In [3]:
# Verification
assert traj.total_tokens > 0
assert len(traj.steps) > 0
print("VERIFICATION PASSED: Phase 14.12 complete")

VERIFICATION PASSED: Phase 14.12 complete
